Introducción
En este caso se trabajó con una base de datos de licencias pendientes. El objetivo fue aplicar un proceso ETL utilizando Python, con el fin de extraer los datos desde un archivo Excel, realizar procesos de limpieza y transformación, calcular nuevas variables relacionadas con los plazos de pronunciamiento y cargar el resultado final en PostgreSQL.

1. Importación de librerías
En esta primera etapa se importan las librerías necesarias para desarrollar el proceso ETL. También se importan herramientas para trabajar con rutas, limpiar nombres de columnas y conectarse con PostgreSQL.

In [170]:
#Se importan las librerías 
import pandas as pd
import numpy as np
import re 
import unicodedata
from pathlib import Path
from sqlalchemy import create_engine, text

2. Definición de la ruta del archivo
Se define la ubicación del archivo Licencias Pendientes.xlsx mediante Path. Luego se verifica que el archivo exista en la carpeta indicada. Si el resultado es True, significa que Python encontró correctamente el archivo.

In [171]:
#Definir ruta 
ruta_excel = Path("Licencias Pendientes.xlsx")

In [172]:
#Verificar que el archivo exista
ruta_excel.exists()

True

3. Revisión de las hojas del Excel
Antes de cargar los datos, se revisan las hojas disponibles dentro del archivo. Esto permite identificar sus nombres exactos y evitar errores al momento de leerlas.

In [173]:
# Revisar las hojas disponibles en el archivo Excel
archivo_excel = pd.ExcelFile(ruta_excel)
archivo_excel.sheet_names

['BASE', 'Feriados']

In [174]:
# Cargar las hojas BASE y Feriados
BASE = pd.read_excel(ruta_excel, sheet_name="BASE")
Feriados = pd.read_excel(ruta_excel, sheet_name="Feriados")

4. Exploración inicial
Después de cargar los datos, se revisan las dimensiones, las primeras filas y la estructura de las tablas.

In [175]:
# Revisar las dimensiones de las tablas cargadas
print("Tabla de licencias pendientes:", BASE.shape)
print("Tabla de feriados:", Feriados.shape)

Tabla de licencias pendientes: (115226, 15)
Tabla de feriados: (488, 1)


5. Limpieza de los nombres de columnas
Los nombres originales de las columnas pueden contener mayúsculas, espacios, tildes, puntos o guiones.
Para facilitar su uso en Python y PostgreSQL, los nombres se transforman a minúsculas, se eliminan las tildes y los espacios se reemplazan por guiones bajos.

In [176]:
# Limpiar y estandarizar los nombres de las columnas de BASE
BASE.columns = [
 unicodedata.normalize("NFKD", columna)
 .encode("ascii", "ignore")
 .decode("utf-8")
 for columna in BASE.columns
]
BASE.columns = (
 pd.Index(BASE.columns)
 .str.strip()
 .str.lower()
 .str.replace(" ", "_", regex=False)
 .str.replace("-", "_", regex=False)
 .str.replace(".", "_", regex=False)
)

In [177]:
# Limpiar y estandarizar los nombres de las columnas de Feriados
Feriados.columns = [
 unicodedata.normalize("NFKD", columna)
 .encode("ascii", "ignore")
 .decode("utf-8")
 for columna in Feriados.columns
]
Feriados.columns = (
 pd.Index(Feriados.columns)
 .str.strip()
 .str.lower()
 .str.replace(" ", "_", regex=False)
 .str.replace("-", "_", regex=False)
 .str.replace(".", "_", regex=False)
)

In [178]:
# Revisar los nombres de las columnas de BASE
BASE.columns

Index(['folio_recepcion', 'fecha_recepcion', 'fecha_resolucion',
       'medico_contralor_usuario', 'proveedor_lme', 'calculado',
       'auditoria_usuario_creacion', 'num_dias', 'protocolo_codigo',
       'visacion', 'rango_dias', 'fecha_maxima_pronunciamiento', 'plazo',
       'visada', 'fuera_plazo'],
      dtype='object')

In [179]:
# Revisar los nombres de las columnas de Feriados
Feriados.columns

Index(['inhabiles_2018'], dtype='object')

6. Revisión de valores nulos
Se utiliza isna().sum() para conocer cuántos valores faltantes existen en cada columna.
Esta revisión permite identificar si hay variables que necesitan un tratamiento especial antes de realizar los cálculos.

In [180]:
# Revisar los valores nulos de la tabla BASE
BASE.isna().sum()

folio_recepcion                    0
fecha_recepcion                    0
fecha_resolucion                3589
medico_contralor_usuario        6794
proveedor_lme                   2128
calculado                          0
auditoria_usuario_creacion         0
num_dias                           0
protocolo_codigo                   6
visacion                        3589
rango_dias                         0
fecha_maxima_pronunciamiento       0
plazo                              0
visada                             0
fuera_plazo                        0
dtype: int64

In [181]:
# Revisar los valores nulos de la tabla Feriados
Feriados.isna().sum()

inhabiles_2018    0
dtype: int64

7. Eliminación de registros duplicados
Se revisan y eliminan las filas completamente repetidas mediante drop_duplicates().
Este procedimiento evita que una misma licencia sea considerada más de una vez y afecte los resultados del análisis.
Después de la eliminación se revisan nuevamente las dimensiones de la tabla.

In [182]:
# Eliminar los registros duplicados de BASE
BASE = BASE.drop_duplicates()

In [183]:
# Revisar las dimensiones después de eliminar duplicados
print("Licencias pendientes sin duplicados:", BASE.shape)

Licencias pendientes sin duplicados: (115226, 15)


8. Conversión de fechas
Las columnas relacionadas con fechas se convierten al tipo datetime mediante pd.to_datetime().
Esto permite realizar comparaciones, calcular diferencias y sumar días hábiles.
El parámetro errors="coerce" transforma en valores nulos las fechas que no pueden ser reconocidas correctamente.

In [184]:
# Convertir las columnas necesarias a formato fecha
BASE["fecha_recepcion"] = pd.to_datetime(BASE["fecha_recepcion"], errors="coerce")
BASE["fecha_resolucion"] = pd.to_datetime(BASE["fecha_resolucion"],errors="coerce")
Feriados["inhabiles_2018"] = pd.to_datetime(Feriados["inhabiles_2018"],errors="coerce")


In [185]:
# Revisar la estructura y los tipos de datos de BASE
BASE.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 115226 entries, 0 to 115225
Data columns (total 15 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   folio_recepcion               115226 non-null  int64         
 1   fecha_recepcion               115226 non-null  datetime64[ns]
 2   fecha_resolucion              111637 non-null  datetime64[ns]
 3   medico_contralor_usuario      108432 non-null  object        
 4   proveedor_lme                 113098 non-null  object        
 5   calculado                     115226 non-null  object        
 6   auditoria_usuario_creacion    115226 non-null  object        
 7   num_dias                      115226 non-null  int64         
 8   protocolo_codigo              115220 non-null  object        
 9   visacion                      111637 non-null  float64       
 10  rango_dias                    115226 non-null  object        
 11  fecha_maxima_

In [186]:
# Visualizar los primeros registros de la tabla
BASE.head()

,folio_recepcion,fecha_recepcion,fecha_resolucion,medico_contralor_usuario,proveedor_lme,calculado,auditoria_usuario_creacion,num_dias,protocolo_codigo,visacion,rango_dias,fecha_maxima_pronunciamiento,plazo,visada,fuera_plazo
0,2019907389,2022-02-10 12:28:11,2022-02-11,CONTRALORIA,NaN,LMN,YMONTANA,7,G2000,1.0,Entre 4 y 11 días,2022-02-14,OK,SI,NO
1,2019907409,2022-03-08 15:10:33,2022-03-09,CONTRALORIA,NaN,LMN,YMONTANA,7,G9003,1.0,Entre 4 y 11 días,2022-03-10,OK,SI,NO
2,2019907410,2022-03-08 12:14:44,2022-03-09,CONTRALORIA,NaN,LMN,YMONTANA,15,G2000,1.0,Mayor a 11 días,2022-03-10,OK,SI,NO
3,2020062308,2021-12-15 17:04:27,2021-12-16,CONTRALORIA,NaN,LMN,MOROSTIC,5,G9003,1.0,Entre 4 y 11 días,2021-12-17,OK,SI,NO
4,2020062309,2021-12-28 10:30:00,2021-12-29,CONTRALORIA,NaN,LMN,MOROSTIC,1,G9015,1.0,Menor igual a 3 días,2021-12-30,OK,SI,NO


9. Clasificación por rango de días
Se crea una nueva columna para clasificar las licencias según el valor de num_dias.
Las condiciones separan los registros en diferentes rangos, lo que facilita resumir y analizar la información.
Para realizar esta clasificación se utiliza np.select().

In [187]:
# Clasificar las licencias según la cantidad de días
BASE["Rango Días Calculado"] = np.select(
    [
        BASE["num_dias"] <= 3,
        (BASE["num_dias"] >= 4) & (BASE["num_dias"] <= 11),
        BASE["num_dias"] > 11
    ],
    [
        "Menor igual a 3 días",
        "Entre 4 y 11 días",
        "Mayor a 11 días"
    ],
    default="Neutro"
)

In [188]:
# Revisar los resultados de la clasificación por rango de días
BASE[["folio_recepcion", "fecha_recepcion", "fecha_resolucion", "medico_contralor_usuario", "proveedor_lme", "calculado", "auditoria_usuario_creacion", "num_dias", "protocolo_codigo", "visacion", "rango_dias", "fecha_maxima_pronunciamiento", "plazo", "visada", "fuera_plazo", "Rango Días Calculado"]].head()

,folio_recepcion,fecha_recepcion,fecha_resolucion,medico_contralor_usuario,proveedor_lme,calculado,auditoria_usuario_creacion,num_dias,protocolo_codigo,visacion,rango_dias,fecha_maxima_pronunciamiento,plazo,visada,fuera_plazo,Rango Días Calculado
0,2019907389,2022-02-10 12:28:11,2022-02-11,CONTRALORIA,NaN,LMN,YMONTANA,7,G2000,1.0,Entre 4 y 11 días,2022-02-14,OK,SI,NO,Entre 4 y 11 días
1,2019907409,2022-03-08 15:10:33,2022-03-09,CONTRALORIA,NaN,LMN,YMONTANA,7,G9003,1.0,Entre 4 y 11 días,2022-03-10,OK,SI,NO,Entre 4 y 11 días
2,2019907410,2022-03-08 12:14:44,2022-03-09,CONTRALORIA,NaN,LMN,YMONTANA,15,G2000,1.0,Mayor a 11 días,2022-03-10,OK,SI,NO,Mayor a 11 días
3,2020062308,2021-12-15 17:04:27,2021-12-16,CONTRALORIA,NaN,LMN,MOROSTIC,5,G9003,1.0,Entre 4 y 11 días,2021-12-17,OK,SI,NO,Entre 4 y 11 días
4,2020062309,2021-12-28 10:30:00,2021-12-29,CONTRALORIA,NaN,LMN,MOROSTIC,1,G9015,1.0,Menor igual a 3 días,2021-12-30,OK,SI,NO,Menor igual a 3 días


In [189]:

# Importar la función para trabajar con días hábiles
from pandas.tseries.offsets import CustomBusinessDay

In [190]:
# Convertir columnas de fecha
BASE["fecha_recepcion"] = pd.to_datetime(BASE["fecha_recepcion"],errors="coerce")

BASE["fecha_resolucion"] = pd.to_datetime(BASE["fecha_resolucion"],errors="coerce")

Feriados["inhabiles_2018"] = pd.to_datetime(Feriados["inhabiles_2018"],errors="coerce")

10. Preparación de los días inhábiles
Se prepara la lista de feriados que será utilizada en el cálculo de días hábiles.
Primero se eliminan los valores nulos, luego se normalizan las fechas y finalmente se eliminan las fechas repetidas.
La lista resultante contiene solo fechas válidas que no deben contarse como días hábiles.

In [191]:
# Crear lista de feriados
feriados = (Feriados["inhabiles_2018"].dropna().drop_duplicates())


In [192]:
# Crear calendario de días hábiles
dia_habil = CustomBusinessDay(weekmask="Mon Tue Wed Thu Fri",holidays=feriados)


11. Asignación de días de pronunciamiento
Se limpia el texto de la columna auditoria_usuario_creacion para evitar diferencias por espacios o uso de mayúsculas y minúsculas.
La asignación se realiza mediante np.where().
Después se aplica la siguiente regla:
Si el usuario es LICMED, se asignan 4 días hábiles.
Si el usuario es distinto de LICMED, se asignan 2 días hábiles.

In [193]:
# Definir plazo
# LICMED = 4 días hábiles
# Otros usuarios = 2 días hábiles
BASE["dias_pronunciamiento"] = np.where(
    BASE["auditoria_usuario_creacion"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.strip() == "LICMED",
    4,
    2
)

12. Cálculo de la fecha máxima de pronunciamiento
La fecha máxima se calcula a partir de fecha_recepcion, sumando los días establecidos para cada registro.
El cálculo considera solamente los días de lunes a viernes y excluye los feriados registrados en la hoja Feriados.
Para realizar este cálculo se utiliza np.busday_offset().
Los registros sin fecha de recepción válida, mantienen un resultado nulo.

In [194]:
# Calcular Fecha Máxima de Pronunciamiento
BASE["fecha_maxima_pronunciamiento_calculado"] = [
    pd.NaT if pd.isna(fecha)
    else fecha + dias * dia_habil
    for fecha, dias in zip(
        BASE["fecha_recepcion"],
        BASE["dias_pronunciamiento"]
    )
]

In [195]:
# Revisar resultado
BASE[["fecha_recepcion","auditoria_usuario_creacion","dias_pronunciamiento","fecha_maxima_pronunciamiento_calculado"]].head()

,fecha_recepcion,auditoria_usuario_creacion,dias_pronunciamiento,fecha_maxima_pronunciamiento_calculado
0,2022-02-10 12:28:11,YMONTANA,2,2022-02-14 12:28:11
1,2022-03-08 15:10:33,YMONTANA,2,2022-03-10 15:10:33
2,2022-03-08 12:14:44,YMONTANA,2,2022-03-10 12:14:44
3,2021-12-15 17:04:27,MOROSTIC,2,2021-12-17 17:04:27
4,2021-12-28 10:30:00,MOROSTIC,2,2021-12-30 10:30:00


In [196]:
BASE["fecha_maxima_pronunciamiento_calculado"] = pd.to_datetime(BASE["fecha_maxima_pronunciamiento_calculado"],errors="coerce")

In [197]:
# Crear condiciones para calcular plazo
condiciones = [
    BASE["fecha_resolucion"].isna(),
    BASE["fecha_maxima_pronunciamiento_calculado"].isna(),
    BASE["fecha_resolucion"] <= BASE["fecha_maxima_pronunciamiento_calculado"],
    BASE["fecha_resolucion"] > BASE["fecha_maxima_pronunciamiento_calculado"]
]

valores = [
    "NO OK",
    "NO OK",
    "SI",
    "NO"
]

In [198]:
BASE["plazo_calculado"] = np.select(
    condiciones,
    valores,
    default="NO OK"
)

13. Cálculo del cumplimiento del plazo
Se compara la fecha de resolución con la fecha máxima de pronunciamiento.
La licencia se considera dentro del plazo cuando la fecha de resolución es menor o igual a la fecha máxima calculada.
Si la resolución ocurre después de esa fecha, se considera fuera del plazo.

In [199]:
# Comparar el plazo original con el plazo calculado
BASE[["fecha_resolucion", "fecha_maxima_pronunciamiento", "plazo", "plazo_calculado"]].head()

,fecha_resolucion,fecha_maxima_pronunciamiento,plazo,plazo_calculado
0,2022-02-11,2022-02-14,OK,SI
1,2022-03-09,2022-03-10,OK,SI
2,2022-03-09,2022-03-10,OK,SI
3,2021-12-16,2021-12-17,OK,SI
4,2021-12-29,2021-12-30,OK,SI


14. Cálculo del estado de visación
Se revisa la columna medico_contralor_usuario.
Cuando existe un médico contralor registrado, la licencia se clasifica como visada. Si el campo está vacío o contiene un valor nulo, se clasifica como no visada.

In [200]:
# Identificar registros sin médico contralor
medico_vacio = (
    BASE["medico_contralor_usuario"].isna() |
    (BASE["medico_contralor_usuario"].astype(str).str.strip() == "")
)

In [201]:
BASE["visada_calculado"] = np.where(
    medico_vacio,
    "NO",
    "SI"
)

In [202]:
# Revisar resultado
BASE[["medico_contralor_usuario","visada_calculado"]].head()

,medico_contralor_usuario,visada_calculado
0,CONTRALORIA,SI
1,CONTRALORIA,SI
2,CONTRALORIA,SI
3,CONTRALORIA,SI
4,CONTRALORIA,SI


In [203]:
# Validar cantidad de casos 
BASE["visada_calculado"].value_counts()

visada_calculado
SI    108432
NO      6794
Name: count, dtype: int64

15. Identificación de licencias fuera de plazo
Se crea la variable fuera_plazo_calculado para identificar las licencias cuya resolución ocurrió después de la fecha máxima permitida.
Esta columna facilita contar y analizar las licencias que no fueron resueltas dentro del plazo establecido.

In [204]:
# Crear columna fuera de plazo
BASE["fuera_plazo_calculado"] = np.where(
    (
        medico_vacio &
        BASE["fecha_resolucion"].notna() &
        BASE["fecha_maxima_pronunciamiento_calculado"].notna() &
        (BASE["fecha_maxima_pronunciamiento_calculado"] < BASE["fecha_resolucion"])
    ),
    "SI",
    "NO"
)

In [205]:
# Revisar resultado
BASE[[ "medico_contralor_usuario","fecha_resolucion","fecha_maxima_pronunciamiento_calculado","fuera_plazo_calculado"]].head()

,medico_contralor_usuario,fecha_resolucion,fecha_maxima_pronunciamiento_calculado,fuera_plazo_calculado
0,CONTRALORIA,2022-02-11,2022-02-14 12:28:11,NO
1,CONTRALORIA,2022-03-09,2022-03-10 15:10:33,NO
2,CONTRALORIA,2022-03-09,2022-03-10 12:14:44,NO
3,CONTRALORIA,2021-12-16,2021-12-17 17:04:27,NO
4,CONTRALORIA,2021-12-29,2021-12-30 10:30:00,NO


16. Tablas resumen
En esta etapa se agrupan los datos para obtener un resumen de los principales resultados del caso. Se revisa la cantidad de licencias según el cumplimiento del plazo, el estado de visación, la resolución fuera de plazo y el rango de días.

In [206]:
# Resumir las licencias según el cumplimiento del plazo
resumen_plazo = (
    BASE
    .groupby("plazo_calculado",as_index=False)
    .agg(cantidad_licencias=("folio_recepcion", "count")
    )
    .sort_values("cantidad_licencias",ascending=False)
)

In [207]:
resumen_plazo

,plazo_calculado,cantidad_licencias
2,SI,111455
1,NO OK,3589
0,NO,182


In [208]:
# Resumir las licencias según su estado de visación
resumen_visacion = (
    BASE
    .groupby("visada_calculado",as_index=False)
    .agg(cantidad_licencias=("folio_recepcion", "count")
    )
    .sort_values("cantidad_licencias",ascending=False)
)

In [209]:
resumen_visacion

,visada_calculado,cantidad_licencias
1,SI,108432
0,NO,6794


In [210]:
# Resumir las licencias resueltas dentro y fuera de plazo
resumen_fuera_plazo = (
    BASE
    .groupby("fuera_plazo_calculado",as_index=False)
    .agg(cantidad_licencias=("folio_recepcion", "count")
    )
    .sort_values("cantidad_licencias",ascending=False)
)

In [211]:
resumen_fuera_plazo

,fuera_plazo_calculado,cantidad_licencias
0,NO,115213
1,SI,13


In [212]:
# Resumir las licencias según el rango de días
resumen_rango_dias = (
    BASE
    .groupby("Rango Días Calculado",as_index=False)
    .agg(cantidad_licencias=("folio_recepcion", "count")
    )
    .sort_values("cantidad_licencias",ascending=False)
)

In [213]:
resumen_rango_dias

,Rango Días Calculado,cantidad_licencias
0,Entre 4 y 11 días,50588
1,Mayor a 11 días,46816
2,Menor igual a 3 días,17822


In [214]:
# Resumir las licencias según los días de pronunciamiento
resumen_dias_pronunciamiento = (
    BASE
    .groupby("dias_pronunciamiento",as_index=False)
    .agg(cantidad_licencias=("folio_recepcion", "count")
    )
    .sort_values("dias_pronunciamiento",ascending=True)
)

In [215]:
resumen_dias_pronunciamiento

,dias_pronunciamiento,cantidad_licencias
0,2,2798
1,4,112428


17. Exportación de tablas resumen
Las tablas resumen se guardan en un archivo Excel, utilizando una hoja diferente para cada resultado.

In [216]:
# Exportar las tablas resumen en un archivo Excel
with pd.ExcelWriter(
    "tablas_resumen_licencias.xlsx",
    engine="openpyxl"
) as writer:

    resumen_plazo.to_excel(
        writer,
        sheet_name="Resumen_Plazo",
        index=False
    )

    resumen_visacion.to_excel(
        writer,
        sheet_name="Resumen_Visacion",
        index=False
    )

    resumen_fuera_plazo.to_excel(
        writer,
        sheet_name="Fuera_Plazo",
        index=False
    )

    resumen_rango_dias.to_excel(
        writer,
        sheet_name="Rango_Dias",
        index=False
    )

    resumen_dias_pronunciamiento.to_excel(
        writer,
        sheet_name="Dias_Pronunciamiento",
        index=False
    )

In [217]:
print("Tablas resumen exportadas correctamente")

Tablas resumen exportadas correctamente


18. Exportación del dataset limpio
Se exporta la tabla limpia y transformada a formatos Excel y CSV. Estos archivos contienen los datos preparados durante el proceso ETL.

In [218]:
# Exportar tablas resumen
with pd.ExcelWriter(carpeta_salida / "tablas_resumen_licencias.xlsx", engine="openpyxl") as writer:
    resumen_plazo.to_excel(writer, sheet_name="Resumen_Plazo", index=False)
    resumen_visacion.to_excel(writer, sheet_name="Resumen_Visacion", index=False)
    resumen_fuera_plazo.to_excel(writer, sheet_name="Fuera_Plazo", index=False)
    resumen_rango_dias.to_excel(writer, sheet_name="Rango_Dias", index=False)
    resumen_dias_pronunciamiento.to_excel(writer, sheet_name="Dias_Pronunciamiento", index=False)

18. Conexión con PostgreSQL
Se definen los datos de conexión: usuario, contraseña, host, puerto y nombre de la base de datos.
Luego se utiliza create_engine() para crear la conexión entre Python y PostgreSQL.

In [ ]:
#Datos de conexión a PostgreSQL
import os

usuario = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST", "localhost")
puerto = os.getenv("DB_PORT", "5432")
base_datos = os.getenv("DB_NAME")

In [220]:
# Crear la conexión
engine = create_engine(
 f"postgresql+psycopg2://{usuario}:{password}@{host}:{puerto}/{base_datos}"
)

In [221]:
# Probar que la conexión a PostgreSQL funcione correctamente
with engine.connect() as conexion:
 resultado = conexion.execute(text("SELECT version();"))
 print(resultado.fetchone())

('PostgreSQL 18.3 on x86_64-windows, compiled by msvc-19.44.35225, 64-bit',)


19. Carga de los datos
La tabla limpia y transformada se carga en PostgreSQL mediante to_sql().
Se utiliza if_exists="replace" para reemplazar la tabla si ya existe, e index=False para evitar que el índice de pandas se guarde como una columna adicional.

In [222]:
# Cargar la tabla en PostgreSQL
BASE.to_sql(
 name="Licencias_Pendientes",
 con=engine,
 schema="public",
 if_exists="replace",
 index=False
)

226

In [223]:
# Confirmar que la tabla fue cargada correctamente
print("Tabla Base de licencias pendientes cargada correctamente en PostgreSQL")

Tabla Base de licencias pendientes cargada correctamente en PostgreSQL


El proceso permitió limpiar y preparar la base de licencias pendientes para su análisis. Además, se calcularon automáticamente los días disponibles para el pronunciamiento, la fecha máxima y el cumplimiento del plazo. Finalmente, los datos fueron cargados en PostgreSQL y exportados a archivos que pueden utilizarse en análisis posteriores.